# 1. 군집화를 위한 데이터 준비

## 1-1. 들어가기 전에

- 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

# as_frame = True로 설정시 판다스 DataFrame으로 반환
    # 기본 값은 False로, 넘파이 배열로 반환
# return_X_y = True로 설정시 특성 행렬과 타겟 벡터를 별도로 반환
df, y = load_wine(as_frame=True, return_X_y=True)
print(df.head())    # 특성 행렬
print(y)    # 타겟 벡터

df['quality'] = y   # 타겟 벡터를 특성 행렬에 추가
print(df.head())

In [ ]:
# df에는 특성 행렬과 타겟 벡터가 모두 포함시켜 놓음
# 특성 행렬과 타겟 벡터를 분리
X = df.drop('quality', axis=1).values   # 특성 행렬
# 이번에는 이진 분류가 아니므로 0과 1로 변환하지 않음
y = df['quality'].values    # 타겟 벡터
print(X.shape, y.shape)     # 178개의 샘플과 13개의 특성, 타겟 벡터는 178개의 값

# 2. 주성분 분석(PCA)과 데이터 군집화
- PCA와 데이터 군집화 과정이 왜 필요할까?
    1. 와인 데이터에는 알코올, 말산, 마그네슘 등 총 13개의 특성이 있음.
    2. 한 번에 13개의 특성을 모두 보면서 데이터의 관계를 파악하기란 불가능에 가까움.
    3. 따라서, PCA를 통해 13개의 특성을 가장 중요한 정보 2개로 압축할 것.
    4. 이후, 군집화 과정을 통해 본래의 와인 등급에 맞게 잘 분류 되는지를 시각적으로 확인할 것.
    
## 2-1. 실습 목표

1. 데이터 차원 축소
2. 데이터 군집화

### 2-1-1. 주성분 분석 (PCA, incipal Component Analysis)

- 데이터의 **중요한 특징**을 유지하면서 차원을 줄이는 기술

1. **주성분**
    - 특성이 얼마나 크게 `분산` 되어 있는가를 기준으로 삼음
        - **분산이 크다:** 어떤 특성의 값이 100~200 사이에 넓게 퍼져있음
        - **분산이 작다:** 어떤 특성의 값이 100~101 사이로 좁게 모여있음.
    - 값이 거의 변하지 않는 특성 (분산이 작은 특성)은 데이터의 차이를 설명하는데 도움이 되지 못함

1. **차원 축소**
    - **주의) 주성분 외 특성을 버리는 것이 아님. `새로운 주 성분`이라는 `축`을 만드는 것.**
    - 데이터의 특성(feature)이 너무 많으면 모델이 복잡해지고, 학습 시간도 오래 걸리며, 때로는 성능이 오히려 나빠질 수도 있음. (차원의 저주 (Curse of Dimension)
    - 이 문제를 해결하기 위해 원래 데이터의 특성들 중, 중요한 데이터 (주성분)은 최대한 보존하면서 특성의 개수를 줄이는 방식을 선택.
        - 불필요한 복잡성도 줄어듦으로써 모델 성능 향상도 기대 할 수 있음.
        - 이 과정에서 중요하지 않은 특성을 제거하며 데이터에 포함된 `불필요한 노이즈`까지 제거하는 부수적 효과도 누릴 수 있음.

In [ ]:
# decomposition: 차원 축소와 관련된 다양한 기능들이 포함되어 있음
from sklearn.decomposition import PCA

# 1. PCA: 특성을 PCA로 분석해 두 개의 Feature로 축소한 X_pca를 생성
'''n_components
- 축소할 차원의 수
- n_components의 값을 0.95와 같이 실수로 지정하면, 
  분산 설명 비용이 95%가 되도록 하는 최소한의 차원을 선택
'''
pca = PCA(n_components=2)    # 2개의 Feature로 축소

# fit_transform: PCA 모델을 데이터에 맞추고, 데이터를 변환
X_pca = pca.fit_transform(X)
print(X.shape)       # (178, 13)
print(X_pca.shape)   # (178, 2)

### 2-1-2. 데이터 군집화 (Clustering)

- 클러스터링은 비지도 학습의 한 종류
    - 정답이 없는 데이터를 토대로 학습한 결과를 도출
- **클러스터링의 목표**
    1. 서로 비슷한 특징을 가진 데이터들을 `그룹`으로 묶어내는 것
    2. 단, 정답(label)이 없기 때문에 오직 특징만 보고 그룹을 만들어야 함.
- **클러스터링 종류**
    1. K-평균 (K-Means Clustering): K개의 그룹으로 묶기 위해, 각 그룹의 평균을 중심점으로 잡아 그룹화
    2. 계층적 클러스터링(Hierarchical Clustering): 데이터들을 계층적으로 묶어나가며 군집을 형성하는 방식
    3. 가우시안 믹스쳐 모델(GMM): 각 데이터 포인트가 각 군집에 속할 확률을 계산

- **K-Means Clustering 작동 방식**
    1. 시작: K개의 중심점을 데이터에 **무작위로 지정**
    2. 그룹화: 모든 데이터 포인트가 가장 **가까운 중심점을 탐색** 후, 해당 그룹에 속하게 함
    3. 중심점 업데이트: 각 그룹에 속한 데이터 포인트들의 **평균 위치**를 다시 계산해서, 해당 위치로 중심점을 옮김
    4. 반복: 중심점이 더 이상 크게 움직이지 않을때 까지 **2번과 3번을 반복**

In [ ]:
# cluster: 군집화와 관련된 다양한 기능들이 포함되어 있음
from sklearn.cluster import KMeans

# 2. K-Means Clustering
# n_clusters: 군집의 수
# random_state: 랜덤 시드 고정
# n_init: K-Means 알고리즘의 초기 중심점을 설정하는 방법
    # 'auto': 기본값으로, 알고리즘이 자동으로 최적의 초기화 방법을 선택
    # 정수값: 지정된 횟수만큼 다른 초기 중심점을 사용하여 군집화를 수행하고, 가장 좋은 결과를 선택
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')    # 와인 등급은 0, 1, 2 3개

# fit_predict: K-Means 모델을 데이터에 맞추고, 각 샘플이 속한 군집 레이블을 반환
clusters = kmeans.fit_predict(X_pca)
print(clusters)   # 각 샘플이 속한 군집 레이블 (0, 1, 2)
# 원본 타겟 벡터 y와 군집 레이블 clusters 비교
print(np.unique(y, return_counts=True))   # 원본 타겟 벡터
print(np.unique(clusters, return_counts=True))   # 군집 레이블

'''
    KMeans 알고리즘은 군집의 중심을 기준으로 데이터를 그룹화하기 때문에,
    군집 레이블이 원본 타겟 벡터와 정확히 일치하지 않을 수 있음.
    예를 들어, 군집 0이 원본 타겟 벡터의 클래스 2에 해당할 수 있음.
    따라서, 군집 레이블과 원본 타겟 벡터 간의 직접적인 비교는 의미가 없을 수 있음.
'''

# 3. 클러스터링 결과 시각화 및 평가

## 3-1. 클러스터링 결과 시각화

- matplotlib을 활용하여 산점도 정보를 출력

In [ ]:

# 3. 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
# cmap: 색상 맵 설정
    # 'viridis', 'plasma', 'inferno', 'magma', 'cividis' 등 다양한 색상 맵이 있음
# s: 마커 크기 설정
# c: 각 점의 색상을 군집 레이블에 따라 다르게 설정

# X_pca는 주요한 두 개의 PCA 특성으로 구성된 2D 배열
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=100)
plt.xlabel('PCA Feature 1')
plt.ylabel('PCA Feature 2')
plt.title('K-Means Clustering of Wine Data after PCA')
plt.colorbar(label='Cluster Label')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# c값을 원본 타겟 벡터 y로 설정하여 시각화
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=100)
plt.xlabel('PCA Feature 1')
plt.ylabel('PCA Feature 2')
plt.title('K-Means Clustering of Wine Data after PCA')
plt.colorbar(label='Cluster Label')
plt.show()

## 3-2. 실제 와인 등급과 비교 분석

- 클러스터링 결과와 실제 와인 등급을 비교해, 왜 K-Means 결과가 실제 등급과 달랐는지 분석
- 이미 정답(label)을 알고 있기 때문에, 이를 활용하여 클러스터링의 성능을 간접적으로 평가가능
    - 본래 비지도 학습 군집화 평가 지표의 경우, 정답이 없음을 가정하여 진행
        - 실루엣 계수, 엘보우 메서드등을 사용

- 이번 실습에서는 간단하게, `c` 색상을 정답 `y`를 사용하여 실제 데이터의 분포를 시각화

## 3-3. 왜 결과에 차이가 나는가?

1. **PCA의 역할**
    - PCA는 13차원 데이터를 2차원으로 압축하는 역할을 성공적으로 수행하였음.
    - 실제 등급 그래프에서도 PCA는 와인 등급을 어느 정도 분리해내는데는 성공하였음.
    - 단, 중앙의 두 그룹은 서로 겹쳐있는 상태로 압축되었음.
2. **K-Means의 한계**
    - K-Means는 군집이 구형이고, 크기가 비슷하다고 가정
    - 그러나 실제 데이터는 서로 겹쳐있고, 불규칙한 형태를 띠고 있음
    - 이 겹쳐진 데이터를 억지로 구형으로 나누려 하기 때문에 실제 등급과 다른 결과가 나올 수 밖에 없음.